# Romenagri: the chain, stage by stage

Copyright (C) 1993-2026 Abhishek Choudhary. SPDX-License-Identifier: GPL-2.0-or-later. Part of the Romenagri Transliteration System.

This notebook runs the **compiled C programs** of `Romenagri/` (`uni2acii`, `acii2rmn`, `rmn2acii`, `acii2uni`) on Devanagari text and shows what each stage writes. Nothing here is a model: the cells call the programs. It needs Python 3, `gcc`, `make` and `flex` (the first cell compiles a copy of `Romenagri/` in a temporary directory); to use programs that are already built, set `ROMENAGRI_BIN` to their directory.

The outputs stored in the notebook are those of the build of the site (`romenagri-site/build_site.py`), which runs every cell.

In [1]:
import os, shutil, struct, subprocess, tempfile, unicodedata

def find_source():
    d = os.path.abspath(os.getcwd())
    for _ in range(4):
        for cand in (os.path.join(d, "Romenagri"), d):
            if os.path.exists(os.path.join(cand, "Makefile")) and os.path.exists(os.path.join(cand, "uni2acii.c")):
                return cand
        d = os.path.dirname(d)
    raise SystemExit("Romenagri/ not found: run the notebook from romenagri-site/notebooks/ of the package, or set ROMENAGRI_SRC")

BIN = os.environ.get("ROMENAGRI_BIN")
if not BIN:
    SRC = os.environ.get("ROMENAGRI_SRC") or find_source()
    BIN = os.path.join(tempfile.mkdtemp(prefix="romenagri-"), "Romenagri")
    shutil.copytree(SRC, BIN)
    built = subprocess.run(["make", "all"], cwd=BIN, capture_output=True, text=True)
    if built.returncode != 0:
        raise SystemExit("make all failed:\n" + (built.stdout + built.stderr)[-1500:])
print("programs:", ", ".join(p for p in ("uni2acii", "acii2rmn", "rmn2acii", "acii2uni") if os.path.exists(os.path.join(BIN, p))))

programs: uni2acii, acii2rmn, rmn2acii, acii2uni


## Helpers

The converters are filters. `uni2acii` reads UTF-16LE, `acii2uni` writes it (with a byte order mark), the other two read and write bytes. The three forms of the nukta letters (`asis`, `nfc`, `nfd`) are chosen with `-f`; `-w` turns the warning layer on.

In [2]:
def run(prog, data, *args):
    env = {k: v for k, v in os.environ.items() if not k.startswith("ROMENAGRI_NUKTA_")}
    return subprocess.run([os.path.join(BIN, prog), *args], input=data, capture_output=True, env=env)

def u2a(text, form="asis", warn=False):
    """Unicode text -> (ACII bytes, the lines that uni2acii wrote on stderr)"""
    p = run("uni2acii", b"\xff\xfe" + text.encode("utf-16-le"), "-f", form, *(["-w"] if warn else []))
    return p.stdout, p.stderr.decode().splitlines()

def a2u(data, form="asis"):
    """ACII bytes -> Unicode text"""
    return run("acii2uni", bytes(data), "-f", form).stdout[2:].decode("utf-16-le").replace("\r\n", "\n")

def a2r(data):
    """ACII bytes -> Romenagri (a str), or None if acii2rmn crashes"""
    p = run("acii2rmn", bytes(data))
    return p.stdout.decode("latin-1") if p.returncode == 0 else None

def r2a(text):
    """Romenagri -> ACII bytes, or None if rmn2acii crashes"""
    p = run("rmn2acii", text.encode("latin-1"))
    return p.stdout if p.returncode == 0 else None

hexb = lambda b: " ".join("%02X" % x for x in b)
cps = lambda s: " ".join("U+%04X" % ord(c) for c in s)

## The four stages on one word

The first word of the Rigveda, *agnimeele*: Unicode, ACII, Romenagri, ACII again, Unicode again.

In [3]:
word = "\u0905\u0917\u094d\u0928\u093f\u092e\u0940\u0933\u0947"
a, _ = u2a(word)
rmn = a2r(a)
back = r2a(rmn)
print("Unicode :", word, "  ", cps(word))
print("ACII    :", hexb(a))
print("Romenagri:", rmn)
print("ACII    :", hexb(back), "(the same bytes)" if back == a else "(NOT the same bytes)")
print("Unicode :", a2u(back), " (the same text)" if a2u(back) == word else " (not the same text)")

Unicode : अग्निमीळे    U+0905 U+0917 U+094D U+0928 U+093F U+092E U+0940 U+0933 U+0947
ACII    : A4 B5 E8 C6 DB CC DC D2 E1
Romenagri: _agnimeelvraee
ACII    : A4 B5 E8 C6 DB CC DC D2 E1 (the same bytes)
Unicode : अग्निमीळे  (the same text)


## The eleven letters with a nukta, in three forms

ACII writes a letter with a nukta as the byte of the consonant and `E9`, the byte of U+093C. Unicode has eleven letters that are such a pair in one code point; `unicode.h` has a row for only two of them. The rule of the nukta layer: a code point that `unicode.h` has no row for and `nukta.h` lists is written as the consonant and `E9`. The *form* decides how the pairs and the single code points are written at the Unicode end: `asis` (the default: what the converters always wrote), `nfc`, `nfd`.

In [4]:
letters = [0x0929, 0x0931, 0x0934] + list(range(0x0958, 0x0960))
print("%-3s %-7s %-22s %-14s %-14s %-14s %s" % ("", "code", "name", "asis", "nfc", "nfd", "NFC composes the pair?"))
for cp in letters:
    ch = chr(cp)
    cols = [hexb(u2a(ch, f)[0]) for f in ("asis", "nfc", "nfd")]
    base = unicodedata.decomposition(ch).split()
    composes = unicodedata.normalize("NFC", chr(int(base[0], 16)) + chr(0x093c)) == ch if base else False
    print("%-3s U+%04X %-22s %-14s %-14s %-14s %s" % (ch, cp, unicodedata.name(ch).replace("DEVANAGARI LETTER ", "").lower(), *cols, "yes" if composes else "no (composition exclusion)"))

    code    name                   asis           nfc            nfd            NFC composes the pair?
ऩ   U+0929 nnna                   C7             C7             C6 E9          yes
ऱ   U+0931 rra                    D0             D0             CF E9          yes
ऴ   U+0934 llla                   D2 E9          D2 E9          D2 E9          yes
क़   U+0958 qa                     B3 E9          B3 E9          B3 E9          no (composition exclusion)
ख़   U+0959 khha                   B4 E9          B4 E9          B4 E9          no (composition exclusion)
ग़   U+095A ghha                   B5 E9          B5 E9          B5 E9          no (composition exclusion)
ज़   U+095B za                     BA E9          BA E9          BA E9          no (composition exclusion)
ड़   U+095C dddha                  BF E9          BF E9          BF E9          no (composition exclusion)
ढ़   U+095D rha                    C0 E9          C0 E9          C0 E9          no (composition exclusion)
फ़   U+095E f

## The other direction

What `acii2uni` writes for the bytes of a nukta letter. In `asis` a pair stays a pair and a byte of its own stays a single code point; `nfc` composes the three pairs that NFC composes; `nfd` writes every letter as a consonant and a nukta.

In [5]:
for bs in ("C7", "D0", "C6 E9", "CF E9", "D2 E9", "BF E9"):
    data = bytes(int(x, 16) for x in bs.split())
    print("%-6s" % bs, "   ".join("%s: %s %-12s" % (f, a2u(data, f), cps(a2u(data, f))) for f in ("asis", "nfc", "nfd")))

C7     asis: ऩ U+0929         nfc: ऩ U+0929         nfd: ऩ U+0928 U+093C
D0     asis: ऱ U+0931         nfc: ऱ U+0931         nfd: ऱ U+0930 U+093C
C6 E9  asis: ऩ U+0928 U+093C   nfc: ऩ U+0929         nfd: ऩ U+0928 U+093C
CF E9  asis: ऱ U+0930 U+093C   nfc: ऱ U+0931         nfd: ऱ U+0930 U+093C
D2 E9  asis: ऴ U+0933 U+093C   nfc: ऴ U+0934         nfd: ऴ U+0933 U+093C
BF E9  asis: ड़ U+0921 U+093C   nfc: ड़ U+0921 U+093C   nfd: ड़ U+0921 U+093C


## The cases C1 to C8

Each case is a place where the layer, or the standard that it follows, asked for a decision. Each is run through the whole chain in the three forms; where the three forms agree, one line is printed. `crash` means that `acii2rmn` ended with a segmentation fault (a halant followed by a nukta, `E8 E9`, is also ISCII's soft halant).

In [6]:
cases = [
    ("C1", "U+095C", "\u095c"),
    ("C1", "da + nukta", "\u0921\u093c"),
    ("C1", "all eight", "".join(chr(c) for c in range(0x0958, 0x0960))),
    ("C2", "U+0929 U+0931 U+0934", "\u0929\u0931\u0934"),
    ("C2", "the three as pairs", "\u0928\u093c\u0930\u093c\u0933\u093c"),
    ("C3", "na, halant, nukta", "\u0928\u094d\u093c"),
    ("C5", "na + e acute", "\u0928\u00e9"),
    ("C6", "U+095C, halant, va", "\u095c\u094d\u0935"),
    ("C6", "da, nukta, halant, va", "\u0921\u093c\u094d\u0935"),
    ("C7", "U+0934", "\u0934"),
    ("C8", "RRA-halant-HA, LA-halant-HA", "\u0931\u094d\u0939\u0932\u094d\u0939"),
]
def chain(text, form):
    a, _ = u2a(text, form)
    rmn = a2r(a)
    if rmn is None:
        return hexb(a), "crash", a2u(a, form)
    back = r2a(rmn)
    return hexb(a), rmn, a2u(back, form)

for cid, label, text in cases:
    res = {f: chain(text, f) for f in ("asis", "nfc", "nfd")}
    groups = []
    for f, r in res.items():
        for g in groups:
            if g[1] == r:
                g[0].append(f); break
        else:
            groups.append(([f], r))
    print("%s  %s  in: %s" % (cid, label, cps(text)))
    for forms, (a, rmn, out) in groups:
        print("      %-10s ACII %-26s Romenagri %-16s out: %s" % ("all three" if len(forms) == 3 else ",".join(forms), a, rmn, cps(out)))

C1  U+095C  in: U+095C
      all three  ACII BF E9                      Romenagri _rda             out: U+0921 U+093C
C1  da + nukta  in: U+0921 U+093C
      all three  ACII BF E9                      Romenagri _rda             out: U+0921 U+093C
C1  all eight  in: U+0958 U+0959 U+095A U+095B U+095C U+095D U+095E U+095F
      all three  ACII B3 E9 B4 E9 B5 E9 BA E9 BF E9 C0 E9 C9 E9 CD E9 Romenagri kzakhzagzajza_rda_rdhaphzayza out: U+0915 U+093C U+0916 U+093C U+0917 U+093C U+091C U+093C U+0921 U+093C U+0922 U+093C U+092B U+093C U+092F U+093C
C2  U+0929 U+0931 U+0934  in: U+0929 U+0931 U+0934
      asis       ACII C7 D0 D2 E9                Romenagri _nza_rralvrza    out: U+0929 U+0931 U+0933 U+093C
      nfc        ACII C7 D0 D2 E9                Romenagri _nza_rralvrza    out: U+0929 U+0931 U+0934
      nfd        ACII C6 E9 CF E9 D2 E9          Romenagri nzarzalvrza      out: U+0928 U+093C U+0930 U+093C U+0933 U+093C
C2  the three as pairs  in: U+0928 U+093C U+0930 U+093C U+0933 U+0

## The warning layer

`uni2acii` writes the low byte of a code point that `unicode.h` has no row for. The low byte of U+00E9 (an e with an acute accent) is `E9`, the nukta byte. With `-w` the converter reports each such code point once, on stderr; the bytes are the same with and without it.

In [7]:
text = "\u0928\u00e9"
a_quiet, err_quiet = u2a(text)
a_warn, err_warn = u2a(text, warn=True)
print("bytes without -w:", hexb(a_quiet), "  stderr:", err_quiet)
print("bytes with -w   :", hexb(a_warn), "  (the same:", a_quiet == a_warn, ")")
print("\n".join(err_warn))

bytes without -w: C6 E9   stderr: []
bytes with -w   : C6 E9   (the same: True )
uni2acii: note: a code point that unicode.h has no row for is written as its low byte, which ACII reads as another character; every converter of this kind does this (TRANSLITERATION.md, section 11.6)
uni2acii: warning: U+00E9 has no ACII byte and is written as its low byte E9, the nukta byte
uni2acii: 1 code point with no ACII byte was written as its low byte (0 further occurrences not listed)


## Choosing the form

The form can be chosen in four ways; the last one given wins: the macro `NUKTA_FORM` when a converter is compiled, the environment variable `ROMENAGRI_NUKTA_FORM`, the flag `-f`/`--form=`, and, in a C program that includes `nukta.h`, the function `nukta_set_form()`. Here the environment variable and the flag:

In [8]:
text = "\u0928\u093c"
def via_env(form, *flags):
    p = subprocess.run([os.path.join(BIN, "uni2acii"), *flags], input=b"\xff\xfe" + text.encode("utf-16-le"), capture_output=True,
                       env={**{k: v for k, v in os.environ.items() if not k.startswith("ROMENAGRI_NUKTA_")}, **({"ROMENAGRI_NUKTA_FORM": form} if form else {})})
    return hexb(p.stdout) if p.returncode == 0 else "error: " + p.stderr.decode().strip()
print("no choice        :", via_env(None))
print("environment: nfc :", via_env("nfc"))
print("environment: nfc, flag -f asis (the flag wins):", via_env("nfc", "-f", "asis"))
print("a name that is not a form:", via_env("nfq"))

no choice        : C6 E9
environment: nfc : C7
environment: nfc, flag -f asis (the flag wins): C6 E9
a name that is not a form: error: uni2acii: ROMENAGRI_NUKTA_FORM is "nfq", which is not a form: use asis, nfc or nfd


## Where to read more

* `Romenagri/TRANSLITERATION.md`: the specification. Section 15 is the nukta layer; section 15.7 is the cases above with the decisions.
* `romenagri-site/docs/`: the pages of the site, with a demonstration in JavaScript that `build_site.py --check` compares with these programs.
* `tests/run_all.sh`: the test suite of the package.